In [32]:
from matchms.importing import load_from_mgf
import matchms.filtering as ms_filters
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import DataStructs
import numpy as np
import pandas as pd
import pickle


In [17]:
pesticides = list(load_from_mgf("/Users/timvandriel/Desktop/CSCI558/project/data/GNPS-COLLECTIONS-PESTICIDES-POSITIVE.mgf"))
print(f"Loaded {len(pesticides)} spectra from pesticides.mgf")

s0 = pesticides[0]
print("\n--- Spectrum 0 Inspection ---")
print("Raw metadata keys:", list(s0.metadata.keys()))
print("Compound Name:    ", s0.get("compound_name", s0.get("name")))
print("SMILES:           ", s0.get("smiles"))
print("Precursor m/z:    ", s0.get("precursor_mz", s0.get("pepmass")))
print("Raw peak count:   ", len(s0.peaks.mz))

pharmaceuticals = list(load_from_mgf("/Users/timvandriel/Desktop/CSCI558/project/data/GNPS-NIH-CLINICALCOLLECTION1.mgf"))
print(f"Loaded {len(pharmaceuticals)} spectra from pharmaceuticals.mgf")

s0 = pharmaceuticals[0]
print("\n--- Spectrum 0 Inspection ---")
print("Raw metadata keys:", list(s0.metadata.keys()))
print("Compound Name:    ", s0.get("compound_name", s0.get("name")))
print("SMILES:           ", s0.get("smiles"))
print("Precursor m/z:    ", s0.get("precursor_mz", s0.get("pepmass")))
print("Raw peak count:   ", len(s0.peaks.mz))

Loaded 653 spectra from pesticides.mgf

--- Spectrum 0 Inspection ---
Raw metadata keys: ['charge', 'ionmode', 'smiles', 'inchi', 'pubmed', 'usi', 'scans', 'ms_level', 'instrument_type', 'file_name', 'peptide_sequence', 'organism_name', 'compound_name', 'principal_investigator', 'data_collector', 'submit_user', 'confidence', 'spectrum_id', 'precursor_mz']
Compound Name:     Pesticide3_3-Hydroxycarbofuran_C12H15NO4_3,7-Benzofurandiol, 2,3-dihydro-2,2-dimethyl-, 7-(methylcarbamate) M+H
SMILES:            CC1(C(C2=C(O1)C(=CC=C2)OC(=O)NC)O)C
Precursor m/z:     238.107
Raw peak count:    30
Loaded 377 spectra from pharmaceuticals.mgf

--- Spectrum 0 Inspection ---
Raw metadata keys: ['charge', 'ionmode', 'smiles', 'usi', 'scans', 'ms_level', 'instrument_type', 'file_name', 'peptide_sequence', 'organism_name', 'compound_name', 'principal_investigator', 'data_collector', 'submit_user', 'confidence', 'spectrum_id', 'precursor_mz']
Compound Name:     BUPROPION HYDROCHLORIDE [M+H]
SMILES:       

In [20]:
def clean_spectrum(spectrum):
    """Cleans metadata and normalizes peaks while preserving baseline noise."""
    # 1. Standardize metadata keys and values
    spectrum = ms_filters.default_filters(spectrum)
    spectrum = ms_filters.repair_inchi_inchikey_smiles(spectrum)
    spectrum = ms_filters.derive_smiles_from_inchi(spectrum)
    spectrum = ms_filters.add_precursor_mz(spectrum)
    
    # 2. Normalize intensities so the base peak equals 1.0
    spectrum = ms_filters.normalize_intensities(spectrum)
    
    # 3. Restrict m/z to chemically plausible mass window
    spectrum = ms_filters.select_by_mz(spectrum, mz_from=10.0, mz_to=1200.0)
    
    # 4. Remove absolute zeros/artifacts while keeping 99.9% of real baseline grass
    spectrum = ms_filters.select_by_relative_intensity(spectrum, intensity_from=0.0001)
    
    return spectrum

cleaned_pesticides = []
dropped_count = 0

for s in pesticides:
    s_clean = clean_spectrum(s)
    # Keep only if spectrum exists and has at least 3 peaks
    if s_clean is not None and len(s_clean.peaks.mz) >= 3:
        cleaned_pesticides.append(s_clean)
    else:
        dropped_count += 1

print(f"Pesticides - Raw: {len(pesticides)}")
print(f"Pesticides - Cleaned & Retained: {len(cleaned_pesticides)}")
print(f"Pesticides - Dropped (< 3 peaks or None): {dropped_count}")

# Check spectrum 0 after cleaning
s0_clean = cleaned_pesticides[0]
print("\n--- Spectrum 0 After Cleaning ---")
print("Peak count:   ", len(s0_clean.peaks.mz))
print("Max intensity:", s0_clean.peaks.intensities.max())
print("Min intensity:", s0_clean.peaks.intensities.min())

cleaned_pharmaceuticals = []
dropped_count = 0

for s in pharmaceuticals:
    s_clean = clean_spectrum(s)
    # Keep only if spectrum exists and has at least 3 peaks
    if s_clean is not None and len(s_clean.peaks.mz) >= 3:
        cleaned_pharmaceuticals.append(s_clean)
    else:
        dropped_count += 1

print(f"Pharmaceuticals - Raw: {len(pharmaceuticals)}")
print(f"Pharmaceuticals - Cleaned & Retained: {len(cleaned_pharmaceuticals)}")
print(f"Pharmaceuticals - Dropped (< 3 peaks or None): {dropped_count}")

# Check spectrum 0 after cleaning
s0_clean = cleaned_pharmaceuticals[0]
print("\n--- Spectrum 0 After Cleaning ---")
print("Peak count:   ", len(s0_clean.peaks.mz))
print("Max intensity:", s0_clean.peaks.intensities.max())
print("Min intensity:", s0_clean.peaks.intensities.min())

Pesticides - Raw: 653
Pesticides - Cleaned & Retained: 653
Pesticides - Dropped (< 3 peaks or None): 0

--- Spectrum 0 After Cleaning ---
Peak count:    30
Max intensity: 1.0
Min intensity: 0.0011080071729030483
Pharmaceuticals - Raw: 377
Pharmaceuticals - Cleaned & Retained: 377
Pharmaceuticals - Dropped (< 3 peaks or None): 0

--- Spectrum 0 After Cleaning ---
Peak count:    1400
Max intensity: 1.0
Min intensity: 0.000442453404125878


In [21]:
def validate_chemical_structures(spectra_list, dataset_name="Dataset"):
    """Validates chemical structures using RDKit and returns a list of valid spectra."""
    valid_spectra = []
    valid_mol = []
    valid_smiles = []
    dropped_entries = []
    
    for idx, s in enumerate(spectra_list):
        smiles = s.get("smiles")
        # Check for missing or invalid SMILES
        if not smiles or str(smiles).strip() in ["", "N/A", "NA", "None"]:
            dropped_entries.append((idx, s.get("compound_name", "Unknown"), "Missing SMILES"))
            continue
        
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            dropped_entries.append((idx, s.get("compound_name", "Unknown"), f"Invalid SMILES: {smiles}"))
            continue
        
        valid_spectra.append(s)
        valid_mol.append(mol)
        valid_smiles.append(smiles)
        
    print(f"\n--- {dataset_name} Structure Validation ---")
    print(f"Total spectra input:   {len(spectra_list)}")
    print(f"Valid chemical graphs: {len(valid_mol)}")
    print(f"Dropped (invalid):     {len(dropped_entries)}")
    
    if dropped_entries:
        print("Dropped summary (first 5):", dropped_entries[:5])
        
    return valid_spectra, valid_mol, valid_smiles

# Validate both datasets
pesticides_spectra_valid, pesticides_mols, pesticides_smiles = validate_chemical_structures(
    cleaned_pesticides, "Pesticides"
)

pharma_spectra_valid, pharma_mols, pharma_smiles = validate_chemical_structures(
    cleaned_pharmaceuticals, "Pharmaceuticals"
)
        
            


--- Pesticides Structure Validation ---
Total spectra input:   653
Valid chemical graphs: 653
Dropped (invalid):     0

--- Pharmaceuticals Structure Validation ---
Total spectra input:   377
Valid chemical graphs: 377
Dropped (invalid):     0


In [23]:
# Generate Morgan fingerprints for the valid pesticide molecules
pesticides_fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) for mol in pesticides_mols]
print("Total valid pesticide molecules:", len(pesticides_fps))

# Inspect the first pesticide molecule and its fingerprint
fp0 = pesticides_fps[0]
mol0 = pesticides_mols[0]

# Find how many bits (out of 2048) are flipped on to 1
on_bits = list(fp0.GetOnBits())

print("\n--- Molecule 0 Fingerprint Inspection ---")
print("Compound:      ", pesticides_spectra_valid[0].get("compound_name"))
print("SMILES:        ", pesticides_smiles[0])
print("Total Bits:    ", fp0.GetNumBits())
print("Active (1) Bits:", len(on_bits))
print("First 10 'On' Bit Indices:", on_bits[:10])

pharma_fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) for mol in pharma_mols]
print("Total valid pharmaceutical molecules:", len(pharma_fps))

# Inspect the first pharmaceutical molecule and its fingerprint
fp0 = pharma_fps[0]
mol0 = pharma_mols[0]

# Find how many bits (out of 2048) are flipped on to 1
on_bits = list(fp0.GetOnBits())

print("\n--- Molecule 0 Fingerprint Inspection ---")
print("Compound:      ", pharma_spectra_valid[0].get("compound_name"))
print("SMILES:        ", pharma_smiles[0])
print("Total Bits:    ", fp0.GetNumBits())
print("Active (1) Bits:", len(on_bits))
print("First 10 'On' Bit Indices:", on_bits[:10])


Total valid pesticide molecules: 653

--- Molecule 0 Fingerprint Inspection ---
Compound:       Pesticide3_3-Hydroxycarbofuran_C12H15NO4_3,7-Benzofurandiol, 2,3-dihydro-2,2-dimethyl-, 7-(methylcarbamate)
SMILES:         CC1(C(C2=C(O1)C(=CC=C2)OC(=O)NC)O)C
Total Bits:     2048
Active (1) Bits: 37
First 10 'On' Bit Indices: [295, 345, 371, 505, 650, 656, 695, 757, 807, 1019]
Total valid pharmaceutical molecules: 377

--- Molecule 0 Fingerprint Inspection ---
Compound:       BUPROPION HYDROCHLORIDE
SMILES:         CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1
Total Bits:     2048
Active (1) Bits: 32
First 10 'On' Bit Indices: [1, 54, 114, 165, 216, 283, 329, 386, 392, 561]


In [27]:
# Compute Tanimoto similarity between the first two pesticide fingerprints
self_sim = DataStructs.TanimotoSimilarity(pesticides_fps[0], pesticides_fps[0])
cross_sim = DataStructs.TanimotoSimilarity(pesticides_fps[0], pesticides_fps[100])  # Example: comparing first two pesticide fingerprints

# Inspect the active bits for the first two pesticide fingerprints
bits_0 = set(pesticides_fps[0].GetOnBits())
bits_1 = set(pesticides_fps[100].GetOnBits())

intersection = len(bits_0 & bits_1)  # Common active bits (substructures)
union = len(bits_0 | bits_1)         # Total unique active bits across both

manual_tanimoto = intersection / union

print("--- Step 4B: Pairwise Sanity Check ---")
print(f"Self-similarity (Molecule 0 vs 0):  {self_sim:.4f}")
print(f"Cross-similarity (Molecule 0 vs 1): {cross_sim:.4f}")
print(f"Manual bit calculation (0 vs 1):    {intersection} / {union} = {manual_tanimoto:.4f}")

print("\nMolecule 0 Compound:", pesticides_spectra_valid[0].get("compound_name"))
print("Molecule 1 Compound:", pesticides_spectra_valid[100].get("compound_name"))

--- Step 4B: Pairwise Sanity Check ---
Self-similarity (Molecule 0 vs 0):  1.0000
Cross-similarity (Molecule 0 vs 1): 0.1148
Manual bit calculation (0 vs 1):    7 / 61 = 0.1148

Molecule 0 Compound: Pesticide3_3-Hydroxycarbofuran_C12H15NO4_3,7-Benzofurandiol, 2,3-dihydro-2,2-dimethyl-, 7-(methylcarbamate)
Molecule 1 Compound: Pesticide4_Cymoxanil_C7H10N4O3_(2E)-2-Cyano-N-(ethylcarbamoyl)-2-(methoxyimino)acetamide


In [36]:
def build_and_save_ground_truth(mols, fps, spectra, prefix="pesticides"):
    n = len(fps)
    print(f"\n{'='*45}")
    print(f"Building {n}x{n} Ground Truth Matrix: {prefix.upper()}")
    print(f"{'='*45}")
    
    # Create an n x n matrix to hold Tanimoto similarity scores
    tanimoto_mat = np.zeros((n, n), dtype=np.float32)
    
    # compute Tanimoto similarity for each pair of fingerprints
    for i in range(n):
        tanimoto_mat[i, :] = DataStructs.BulkTanimotoSimilarity(fps[i], fps)
    
    # assert matrix validity
    assert np.allclose(np.diag(tanimoto_mat), 1.0), "Diagonal must be 1.0 (self-identity)"
    assert np.allclose(tanimoto_mat, tanimoto_mat.T), "Matrix must be symmetric"
    print("✓ Matrix validation passed (symmetric, unit diagonal).")
    
    # Build metadata DataFrame
    records = []
    for idx, s in enumerate(spectra):
        records.append({
            "matrix_index": idx,
            "spectrum_id": s.get("spectrum_id", f"{prefix}_{idx}"),
            "compound_name": s.get("compound_name", s.get("name", "Unknown")),
            "precursor_mz": s.get("precursor_mz"),
            "adduct": s.get("adduct", "Unknown"),
            "instrument_type": s.get("instrument_type", "Unknown"),
            "smiles": s.get("smiles"),
            "peak_count": len(s.peaks.mz)
        })
    df_meta = pd.DataFrame(records)
    
    # save outputs to disk
    matrix_file = f"/Users/timvandriel/Desktop/CSCI558/project/data/{prefix}_tanimoto_matrix.npy"
    meta_file = f"/Users/timvandriel/Desktop/CSCI558/project/data/{prefix}_metadata.csv"
    spectra_file = f"/Users/timvandriel/Desktop/CSCI558/project/data/{prefix}_clean_spectra.pkl"
    
    np.save(matrix_file, tanimoto_mat)
    df_meta.to_csv(meta_file, index=False)
    with open(spectra_file, "wb") as f:
        pickle.dump(spectra, f)
    
    print(f"Artifacts saved:")
    print(f"  • {matrix_file}  [shape: {tanimoto_mat.shape}]")
    print(f"  • {meta_file}     [{len(df_meta)} rows]")
    print(f"  • {spectra_file}  [{len(spectra)} spectra]")
    
    off_diag = tanimoto_mat[np.triu_indices_from(tanimoto_mat, k=1)]
    print(f"Off-diagonal Tanimoto summary -> Mean: {off_diag.mean():.3f}, Median: {np.median(off_diag):.3f}, Max: {off_diag.max():.3f}")
    
    return tanimoto_mat, df_meta

In [37]:
pest_matrix, pest_meta = build_and_save_ground_truth(
    pesticides_mols, pesticides_fps, pesticides_spectra_valid, prefix="pesticides"
)


Building 653x653 Ground Truth Matrix: PESTICIDES
✓ Matrix validation passed (symmetric, unit diagonal).
Artifacts saved:
  • /Users/timvandriel/Desktop/CSCI558/project/data/pesticides_tanimoto_matrix.npy  [shape: (653, 653)]
  • /Users/timvandriel/Desktop/CSCI558/project/data/pesticides_metadata.csv     [653 rows]
  • /Users/timvandriel/Desktop/CSCI558/project/data/pesticides_clean_spectra.pkl  [653 spectra]
Off-diagonal Tanimoto summary -> Mean: 0.122, Median: 0.104, Max: 1.000


In [38]:
pharma_matrix, pharma_meta = build_and_save_ground_truth(
    pharma_mols, pharma_fps, pharma_spectra_valid, prefix="pharmaceuticals"
)


Building 377x377 Ground Truth Matrix: PHARMACEUTICALS
✓ Matrix validation passed (symmetric, unit diagonal).
Artifacts saved:
  • /Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_tanimoto_matrix.npy  [shape: (377, 377)]
  • /Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_metadata.csv     [377 rows]
  • /Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_clean_spectra.pkl  [377 spectra]
Off-diagonal Tanimoto summary -> Mean: 0.107, Median: 0.100, Max: 1.000
